# Combined Access Score and Network Threshold Analysis

**Tess Vu**

This notebook merges three analytical streams into a single SAL-level dataset:

1. **2SFCA accessibility scores** (`all_access_w_snap.csv`): Ai_walk, Ai_drive, and snap distances.
2. **k-nearest pharmacy distances** (`sal_pharmacy_distances_k3.csv`): Euclidean, walk, and drive for k=1,2,3.
3. **Threshold exceedance flags** (`sal_threshold_flags_k3.csv`): Binary flags at policy-relevant thresholds.

The unified table is exported as a GeoJSON and a CSV for downstream mapping and app integration.

## SETUP AND IMPORTS

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

In [ ]:
from pathlib import Path

ROOT = Path("..").resolve()          # repository root; notebooks run from notebooks/
DATA = ROOT / "data"
RAW = DATA / "raw"
INTERMEDIATE = DATA / "intermediate"
FINAL = DATA / "final"

IN_ALL_ACCESS_SNAP = INTERMEDIATE / "all_access_w_snap.csv"
IN_DISTANCES = INTERMEDIATE / "sal_pharmacy_distances_k3.csv"
IN_THRESHOLD_FLAGS = INTERMEDIATE / "sal_threshold_flags_k3.csv"
IN_SAL_DEDUP = INTERMEDIATE / "sal_w_ward_dedup" / "sal_w_ward_dedup.shp"
IN_PHARMACIES = RAW / "PHARMACIES_MASTER_FINAL.csv"
IN_DATA_DICTIONARY = ROOT / "data_dictionary.csv"

OUT_SAL_COMBINED_GEOJSON = FINAL / "sal_combined_access.geojson"
OUT_SAL_COMBINED_CSV = FINAL / "sal_combined_access.csv"
OUT_PHARMACIES_GEOJSON = FINAL / "pharmacies.geojson"


## CONFIGURATION

In [ ]:
# POLICY THRESHOLDS FOR ACCESS TYPOLOGY.
# These match the reference thresholds used in the 2SFCA and threshold notebooks.
WALK_THRESHOLD_KM = 3
DRIVE_THRESHOLD_KM = 10

# SNAP DISTANCE FLAG THRESHOLD.
SNAP_FLAG_M = 500

print(f"WALK THRESHOLD: {WALK_THRESHOLD_KM} km")
print(f"DRIVE THRESHOLD: {DRIVE_THRESHOLD_KM} km")
print(f"SNAP FLAG: >{SNAP_FLAG_M} m")


## LOAD SOURCE DATA

In [ ]:
# Load the three source tables.
all_access = pd.read_csv(IN_ALL_ACCESS_SNAP, low_memory = False)
pharmacy_dist = pd.read_csv(IN_DISTANCES)
threshold_flag = pd.read_csv(IN_THRESHOLD_FLAGS)

print(f"ALL_ACCESS: {all_access.shape[0]} rows, {all_access.shape[1]} cols")
print(f"PHARMACY_DIST: {pharmacy_dist.shape[0]} rows, {pharmacy_dist.shape[1]} cols")
print(f"THRESHOLD_FLAG: {threshold_flag.shape[0]} rows, {threshold_flag.shape[1]} cols")

In [ ]:
# Load SAL geometry for the final GeoJSON.
sal_geo = gpd.read_file(IN_SAL_DEDUP)
sal_geo["EA_CODE"] = pd.to_numeric(sal_geo["EA_CODE"], errors = "coerce").astype("Int64")

print(f"SAL GEOMETRY: {sal_geo.shape[0]} features, CRS: {sal_geo.crs}")

## EA_CODE ALIGNMENT CHECK

In [ ]:
# Normalize EA_CODE types across all three tables.
all_access["EA_CODE"] = pd.to_numeric(all_access["EA_CODE"], errors = "coerce").astype("Int64")
pharmacy_dist["EA_CODE"] = pd.to_numeric(pharmacy_dist["EA_CODE"], errors = "coerce").astype("Int64")
threshold_flag["EA_CODE"] = pd.to_numeric(threshold_flag["EA_CODE"], errors = "coerce").astype("Int64")

# Set comparison.
ai_codes = set(all_access["EA_CODE"].dropna())
dist_codes = set(pharmacy_dist["EA_CODE"].dropna())
flag_codes = set(threshold_flag["EA_CODE"].dropna())
geo_codes = set(sal_geo["EA_CODE"].dropna())

all_four = ai_codes & dist_codes & flag_codes & geo_codes

print("EA_CODE ALIGNMENT")
print(f"  all_access:     {len(ai_codes)}")
print(f"  pharmacy_dist:  {len(dist_codes)}")
print(f"  threshold_flag: {len(flag_codes)}")
print(f"  sal_geometry:   {len(geo_codes)}")
print(f"  intersection:   {len(all_four)}")
print()

# Report any orphans.
only_ai = ai_codes - dist_codes
only_dist = dist_codes - ai_codes
only_flag = flag_codes - ai_codes
print(f"  In all_access only:     {len(only_ai)}")
print(f"  In pharmacy_dist only:  {len(only_dist)}")
print(f"  In threshold_flag only: {len(only_flag)}")

## SNAP DISTANCE ANALYSIS

Before merging, profile the snap distances appended by the snap_distance_append
notebook. Snap distance is the straight-line gap between each SAL centroid and
the network node it was snapped to, which is a proxy for OSM coverage quality at that
location.

In [ ]:
# Provincial summary.
snap_cols = ["walk_snap_dist_m", "drive_snap_dist_m"]

for col in snap_cols:
    mode = col.split("_")[0].upper()
    print(f"{mode} SNAP DISTANCES (meters)")
    for prov in ["Gauteng", "KwaZulu-Natal"]:
        vals = all_access.loc[all_access["PR_NAME"] == prov, col].dropna()
        flagged = (vals > SNAP_FLAG_M).sum()
        print(f"  {prov}: median={vals.median():.0f}, mean={vals.mean():.0f}, "
              f"p95={vals.quantile(0.95):.0f}, max={vals.max():.0f}, "
              f"flagged >{SNAP_FLAG_M}m: {flagged} ({flagged / len(vals) * 100:.1f}%)")
    print()

In [ ]:
# Settlement type breakdown of snap flags.
# This confirms where OSM coverage gaps concentrate.
snap_profile = all_access[["EA_CODE", "PR_NAME", "EA_GTYPE",
                           "walk_snap_dist_m", "drive_snap_dist_m"]].copy()

snap_profile["walk_snap_flag"] = snap_profile["walk_snap_dist_m"] > SNAP_FLAG_M
snap_profile["drive_snap_flag"] = snap_profile["drive_snap_dist_m"] > SNAP_FLAG_M

# Snap flag rate by province and settlement type.
for flag_col in ["walk_snap_flag", "drive_snap_flag"]:
    mode = flag_col.split("_")[0].upper()
    print(f"{mode} SNAP FLAG RATE BY SETTLEMENT TYPE")
    summary = (
        snap_profile
        .groupby(["PR_NAME", "EA_GTYPE"])[flag_col]
        .agg(n_flagged = "sum", n_total = "count")
        .assign(pct = lambda x: (x["n_flagged"] / x["n_total"] * 100).round(1))
    )
    print(summary.to_string())
    print()

**Snap distance takeaway.** OSM coverage gaps concentrate in exactly the SAL
categories that show the worst pharmacy access (Farms and Traditional in KZN).
This means accessibility metrics for those areas are simultaneously the most
policy-relevant and the most uncertain. The snap flag column provides a
per-SAL data quality indicator for downstream consumers.

## SELECT AND MERGE COLUMNS

Build a clean combined table with the minimum columns needed for analysis
and mapping. Avoid carrying forward the full 89-column all_access table.

In [ ]:
# AGE COHORT BINNING.
# Convert 2011 age cohort columns from string to numeric, then collapse
# 17 single-cohort columns into five pharmacy-relevant age bands.
# Source columns are dropped after binning; pop_total_2011 is retained
# as a cross-check against sal2011_pop (small differences expected due
# to the population imputation applied in newpred_clean).

AGE_COLS = [
    "F0_4", "F5_9", "F10_14",
    "F15_19", "F20_24", "F25_29", "F30_34",
    "F35_39", "F40_44", "F45_49",
    "F50_54", "F55_59", "F60_64",
    "F65_69", "F70_74", "F75_79", "F80_84", "F85_",
]

# Coerce to numeric, stored as str in the SAL shapefile source.
for col in AGE_COLS:
    all_access[col] = pd.to_numeric(all_access[col], errors = "coerce")

# Build age bands.
all_access["pop_0_14"] = all_access[["F0_4", "F5_9", "F10_14"]].sum(axis = 1, min_count = 1)
all_access["pop_15_34"] = all_access[["F15_19", "F20_24", "F25_29", "F30_34"]].sum(axis = 1, min_count = 1)
all_access["pop_35_49"] = all_access[["F35_39", "F40_44", "F45_49"]].sum(axis = 1, min_count = 1)
all_access["pop_50_64"] = all_access[["F50_54", "F55_59", "F60_64"]].sum(axis = 1, min_count = 1)
all_access["pop_65plus"] = all_access[["F65_69", "F70_74", "F75_79", "F80_84", "F85_"]].sum(axis = 1, min_count = 1)
all_access["pop_total_2011"] = all_access[AGE_COLS].sum(axis = 1, min_count = 1)

# Validation: confirm band totals equal pop_total_2011 within rounding.
band_cols = ["pop_0_14", "pop_15_34", "pop_35_49", "pop_50_64", "pop_65plus"]
recon = all_access[band_cols].sum(axis = 1)
max_discrepancy = (recon - all_access["pop_total_2011"]).abs().max()
print(f"MAX BAND RECONCILIATION DISCREPANCY: {max_discrepancy:.2f} (expect 0.0)")

# Drop the 17 source columns, bands replace them.
all_access = all_access.drop(columns = AGE_COLS)

print(f"AGE BANDS CREATED: {band_cols}")
print(all_access[band_cols + ["pop_total_2011"]].describe().round(1).to_string())

In [ ]:
# COLUMN RENAMES.
# Align source column names from the SAL shapefile to the final output schema.
# Black_Afri and Indian_or are shapefile-truncated names; renamed for clarity.
all_access = all_access.rename(columns = {
    "Black_Afri": "Black_African",
    "Indian_or":  "Indian_Asian",
})

print("COLUMNS RENAMED:")
print(" Black_Afri to Black_African")
print(" Indian_or to Indian_Asian")

In [ ]:
# EA_TYPE VALUE REPLACEMENT.
# "Formal residential" relabeled to "Suburb" for interpretability.
# The Stats SA label is accurate but not meaningful to non-technical audiences.
# All other EA_TYPE values are retained as-is.
n_replaced = (all_access["EA_TYPE"] == "Formal residential").sum()
all_access["EA_TYPE"] = all_access["EA_TYPE"].replace("Formal residential", "Suburb")

print(f"EA_TYPE REPLACED: 'Formal residential' -> 'Suburb' ({n_replaced:,} rows)")
print()
print("CURRENT EA_TYPE DISTRIBUTION:")
print(all_access["EA_TYPE"].value_counts().to_string())

In [ ]:
# COLUMNS FROM ALL_ACCESS (2SFCA SCORES + DEMOGRAPHICS + SNAP DISTANCES).
access_cols = [
    "EA_CODE", "PR_NAME", "EA_GTYPE", "EA_TYPE", "econ_status",
    "WardID", "sal2011_pop", "sal2023_est", "growth_rate", "area_km2",
    "SP_CODE", "SP_NAME", "MP_CODE", "MP_NAME",
    "MN_MDB_C", "MN_CODE", "MN_NAME", "MN_TYPE",
    "DC_MDB_C", "DC_MN_C", "DC_NAME",
    "Black_African", "White", "Coloured", "Indian_Asian", "Other",
    "pop_0_14", "pop_15_34", "pop_35_49", "pop_50_64", "pop_65plus",
    "pop_total_2011",
    "num_houses", "num_build",
    "Ai_walk", "Ai_drive",
    "walk_rank", "drive_rank",
    "walk_log", "drive_log",
    "walk_snap_dist_m", "drive_snap_dist_m",
]

# Filter to columns that exist.
access_cols = [c for c in access_cols if c in all_access.columns]
access_df = all_access[access_cols].copy()

print(f"ACCESS TABLE: {access_df.shape}")
print(f"Columns: {list(access_df.columns)}")


In [ ]:
# COLUMNS FROM PHARMACY_DIST (K-NEAREST DISTANCES).
# Drop columns already in access_df to avoid duplicates on merge.
dist_drop = ["PR_NAME", "WardID", "sal2023_est"]
dist_cols = [c for c in pharmacy_dist.columns if c not in dist_drop]
dist_df = pharmacy_dist[dist_cols].copy()

print(f"DISTANCE TABLE: {dist_df.shape}")
print(f"Columns: {list(dist_df.columns)}")

In [ ]:
# COLUMNS FROM THRESHOLD_FLAG (EXCEEDANCE FLAGS ONLY).
# Keep only EA_CODE and the boolean flag columns; distances and
# demographic attributes are already present in the other tables.
flag_cols = ["EA_CODE"] + [c for c in threshold_flag.columns if c.startswith("exceeds_")]
flag_df = threshold_flag[flag_cols].copy()

print(f"FLAG TABLE: {flag_df.shape}")
print(f"Flag columns: {len(flag_cols) - 1}")

In [ ]:
# MERGE: access + distance + flags on EA_CODE.
combined = (
    access_df
    .merge(dist_df, on = "EA_CODE", how = "inner")
    .merge(flag_df, on = "EA_CODE", how = "inner")
)

print(f"COMBINED TABLE: {combined.shape[0]} rows, {combined.shape[1]} columns")
print(f"Column count breakdown:")
print(f"  From access: {len(access_cols)}")
print(f"  From distance: {len(dist_cols) - 1}")
print(f"  From flags: {len(flag_cols) - 1}")
print(f"  Total: {combined.shape[1]}")

## SNAP DISTANCE QUALITY FLAGS

Add boolean columns indicating whether a SAL's centroid snapped more than
500m from a network node. High snap distance means the routing and 2SFCA
results for that SAL carry elevated uncertainty.

In [ ]:
# Add quality flag columns.
combined["walk_snap_flag"] = combined["walk_snap_dist_m"] > SNAP_FLAG_M
combined["drive_snap_flag"] = combined["drive_snap_dist_m"] > SNAP_FLAG_M

walk_flagged = combined["walk_snap_flag"].sum()
drive_flagged = combined["drive_snap_flag"].sum()

print(f"WALK SNAP FLAGGED: {walk_flagged} SALs ({walk_flagged / len(combined) * 100:.1f}%)")
print(f"DRIVE SNAP FLAGGED: {drive_flagged} SALs ({drive_flagged / len(combined) * 100:.1f}%)")

## ACCESS TYPOLOGY

Classify each SAL into one of six Access Typology classes by crossing the
2SFCA score (Ai) with the k=1 network distance at the policy threshold.
This produces an Access Typology for both walk and drive modes.

| | Distance < threshold | Distance >= threshold |
|---|---|---|
| **Ai = 0** | Connectivity Gap | **Pharmacy Desert** |
| **Ai > 0, bottom tercile** | Demand Overcrowding | Access Gap |
| **Ai > 0, top tercile** | Well-Served | Artifact Zone |


In [ ]:
def classify_access(row, ai_col, dist_col, threshold_km, tercile_low, tercile_high):
    """Assign a SAL to one of six Access Typology classes.

    Categories combine 2SFCA score tiers (zero, low, high) with
    a binary distance threshold to produce interpretable policy labels.
    """
    ai = row[ai_col]
    dist = row[dist_col]

    # Handle NaN distance (unreachable on network).
    if pd.isna(dist):
        if ai == 0:
            return "Pharmacy desert"
        elif ai <= tercile_low:
            return "Access gap"
        else:
            return "Artifact zone"

    near = dist < threshold_km

    if ai == 0 and near:
        return "Connectivity gap"
    elif ai == 0 and not near:
        return "Pharmacy desert"
    elif ai <= tercile_low and near:
        return "Demand overcrowding"
    elif ai <= tercile_low and not near:
        return "Access gap"
    elif near:
        return "Well-served"
    else:
        return "Artifact zone"

In [ ]:
# Compute province-specific terciles for Ai scores.
# Only use nonzero Ai values for tercile calculation to avoid
# the zero-inflation skewing the thresholds.
for prov in ["Gauteng", "KwaZulu-Natal"]:
    mask = combined["PR_NAME"] == prov

    for mode, ai_col, dist_col, threshold in [
        ("walk", "Ai_walk", "walk_dist_k1_km", WALK_THRESHOLD_KM),
        ("drive", "Ai_drive", "drive_dist_k1_km", DRIVE_THRESHOLD_KM),
    ]:
        # Tercile boundaries from nonzero Ai values only.
        nonzero_ai = combined.loc[mask & (combined[ai_col] > 0), ai_col]

        if len(nonzero_ai) == 0:
            tercile_low = 0
            tercile_high = 0
        else:
            tercile_low = nonzero_ai.quantile(1 / 3)
            tercile_high = nonzero_ai.quantile(2 / 3)

        typology_col = f"{mode}_typology"

        combined.loc[mask, typology_col] = combined.loc[mask].apply(
            classify_access,
            axis = 1,
            ai_col = ai_col,
            dist_col = dist_col,
            threshold_km = threshold,
            tercile_low = tercile_low,
            tercile_high = tercile_high,
        )

        print(f"{prov} {mode.upper()} TERCILES: low={tercile_low:.4f}, high={tercile_high:.4f}")

print()

In [ ]:
# Typology distribution.
for mode in ["walk", "drive"]:
    col = f"{mode}_typology"
    print(f"{mode.upper()} ACCESS TYPOLOGY")
    for prov in ["Gauteng", "KwaZulu-Natal"]:
        prov_df = combined[combined["PR_NAME"] == prov]
        counts = prov_df[col].value_counts()
        pcts = (counts / len(prov_df) * 100).round(1)
        print(f"  {prov}:")
        for cat in ["Well-served", "Demand overcrowding", "Connectivity gap",
                    "Access gap", "Pharmacy desert", "Artifact zone"]:
            n = counts.get(cat, 0)
            p = pcts.get(cat, 0.0)
            print(f"    {cat:25s} {n:6d} ({p:5.1f}%)")
    print()

In [ ]:
# Population-weighted typology distribution.
for mode in ["walk", "drive"]:
    col = f"{mode}_typology"
    print(f"{mode.upper()} ACCESS TYPOLOGY (population-weighted)")
    for prov in ["Gauteng", "KwaZulu-Natal"]:
        prov_df = combined[combined["PR_NAME"] == prov]
        total_pop = prov_df["sal2023_est"].sum()
        pop_by_type = prov_df.groupby(col)["sal2023_est"].sum()
        pct_by_type = (pop_by_type / total_pop * 100).round(1)
        print(f"  {prov} (total pop: {total_pop:,.0f}):")
        for cat in ["Well-served", "Demand overcrowding", "Connectivity gap",
                    "Access gap", "Pharmacy desert", "Artifact zone"]:
            pop = pop_by_type.get(cat, 0)
            p = pct_by_type.get(cat, 0.0)
            print(f"    {cat:25s} {pop:12,.0f} ({p:5.1f}%)")
    print()

## SNAP FLAG CROSS-TAB WITH TYPOLOGY

Check whether high-snap SALs disproportionately fall into Pharmacy Desert
or Connectivity Gap classes. If so, these classes carry a data
quality asterisk.

In [ ]:
for mode in ["walk", "drive"]:
    typology_col = f"{mode}_typology"
    snap_col = f"{mode}_snap_flag"

    print(f"{mode.upper()} TYPOLOGY x SNAP FLAG")

    cross = (
        combined
        .groupby([typology_col, snap_col])
        .size()
        .unstack(fill_value = 0)
    )

    # Rename columns for readability.
    cross.columns = ["Clean (<= 500m)", f"Flagged (> {SNAP_FLAG_M}m)"]
    cross["Flag Rate"] = (
        cross[f"Flagged (> {SNAP_FLAG_M}m)"] /
        (cross["Clean (<= 500m)"] + cross[f"Flagged (> {SNAP_FLAG_M}m)"]) * 100
    ).round(1)

    print(cross.to_string())
    print()

## FINAL COLUMN INVENTORY

In [ ]:
print(f"FINAL TABLE: {combined.shape[0]} rows, {combined.shape[1]} columns")
print()
print("COLUMN INVENTORY:")
for i, col in enumerate(combined.columns):
    dtype = combined[col].dtype
    n_valid = combined[col].notna().sum()
    print(f"  {i:3d}  {col:40s}  {str(dtype):10s}  {n_valid:6d} non-null")

In [ ]:
# REORGANIZE COLUMNS INTO LOGICAL GROUPS.
# Drops _m duplicates for pharmacy distances (snap distances stay in meters,
# circuity and centroid coords are unaffected). Moves typology and scores
# forward, co-locates demographics, groups snap distances with snap flags.

cols_identifiers = [
    "EA_CODE", "PR_NAME", "WardID",
    "EA_GTYPE", "EA_TYPE", "econ_status",
]

cols_geography = [
    "SP_CODE", "SP_NAME",
    "MP_CODE", "MP_NAME",
    "MN_MDB_C", "MN_CODE", "MN_NAME", "MN_TYPE",
    "DC_MDB_C", "DC_MN_C", "DC_NAME",
]

cols_population = [
    "sal2011_pop", "sal2023_est", "growth_rate", "area_km2",
]

cols_demographics = [
    "Black_African", "White", "Coloured", "Indian_Asian", "Other",
    "pop_0_14", "pop_15_34", "pop_35_49", "pop_50_64", "pop_65plus",
    "pop_total_2011",
    "num_houses", "num_build",
]

# Primary analytical outputs first.
cols_primary_outputs = [
    "walk_typology", "drive_typology",
    "Ai_walk", "Ai_drive",
    "walk_rank", "drive_rank",
    "walk_log", "drive_log",
]

# k=1 distances across all modes (reachability).
cols_k1_distances = [
    "walk_dist_k1_km", "drive_dist_k1_km", "euclidean_dist_k1_km",
    "walk_circuity_k1", "drive_circuity_k1",
]

# k=2 and k=3 distances (redundancy and choice).
cols_k2_distances = [
    "walk_dist_k2_km", "drive_dist_k2_km", "euclidean_dist_k2_km",
]
cols_k3_distances = [
    "walk_dist_k3_km", "drive_dist_k3_km", "euclidean_dist_k3_km",
]

# Snap quality: distances and flags together.
cols_snap = [
    "walk_snap_dist_m", "drive_snap_dist_m",
    "walk_snap_flag", "drive_snap_flag",
]

# Threshold exceedance flags grouped by k level.
cols_flags_k1 = sorted([c for c in combined.columns if c.startswith("exceeds_") and "_k1_" in c])
cols_flags_k2 = sorted([c for c in combined.columns if c.startswith("exceeds_") and "_k2_" in c])
cols_flags_k3 = sorted([c for c in combined.columns if c.startswith("exceeds_") and "_k3_" in c])

# Centroid coordinates last for mapping support, not analytical.
cols_centroid = ["centroid_lat", "centroid_lng"]

col_order = (
    cols_identifiers
    + cols_geography
    + cols_population
    + cols_demographics
    + cols_primary_outputs
    + cols_k1_distances
    + cols_k2_distances
    + cols_k3_distances
    + cols_snap
    + cols_flags_k1
    + cols_flags_k2
    + cols_flags_k3
    + cols_centroid
)

# Filter to columns that actually exist.
col_order = [c for c in col_order if c in combined.columns]

# Identify what was dropped.
dropped = [c for c in combined.columns if c not in col_order]

print(f"COLUMNS RETAINED: {len(col_order)}")
print(f"COLUMNS DROPPED: {len(dropped)}")
for d in dropped:
    print(f"  - {d}")

combined = combined[col_order].copy()
print(f"FINAL SHAPE: {combined.shape}")


## BUILD FINAL DATASET

The final dataset's short field names, column order, and three-tier rule
come from the root data dictionary, whose row order is the final column
order. This section reproduces the retired rename script's contract
inside the notebook, so NB8 is the single producer of `data/final/`
(ADR 0001). Each check below raises with a clear message instead of
writing a bad file.


In [ ]:
# DICTIONARY CONTRACT: the full-name columns of the combined table must
# equal the dictionary's full_name list minus the two three-tier columns,
# which this notebook derives. Any difference raises.
dictionary = pd.read_csv(IN_DATA_DICTIONARY)

TIER_COLUMNS = ["walk_3_typology", "drive_3_typology"]
expected_full = [name for name in dictionary["full_name"] if name not in TIER_COLUMNS]

missing = sorted(set(expected_full) - set(combined.columns))
unexpected = sorted(set(combined.columns) - set(expected_full))
if missing or unexpected:
    raise ValueError(
        f"Combined columns differ from the data dictionary.\n"
        f"  missing: {missing}\n  not in dictionary: {unexpected}"
    )

if combined["EA_CODE"].duplicated().any():
    raise ValueError(
        f"Combined table has {combined['EA_CODE'].duplicated().sum():,} duplicate EA_CODE values"
    )

print(f"DICTIONARY CONTRACT: {len(expected_full)} full-name columns match")
print(f"EA_CODE UNIQUE: {combined['EA_CODE'].nunique():,} rows")


In [ ]:
# THREE-TIER COLUMNS (the retired rename script's rule).
# Pharmacy Desert = zero score or at or below the 1/3 quantile of nonzero
# scores; Moderate Access = above 1/3 up to the 2/3 quantile; Served =
# above 2/3. Terciles are computed within each province from nonzero
# scores only, using exact 1/3 and 2/3 quantiles.
TIER_AI = {"walk_3_typology": "Ai_walk", "drive_3_typology": "Ai_drive"}

for tier_col, ai_col in TIER_AI.items():
    combined[tier_col] = "Pharmacy Desert"
    for _, idx in combined.groupby("PR_NAME").groups.items():
        ai = combined.loc[idx, ai_col]
        nonzero = ai[ai > 0]
        if nonzero.empty:
            continue
        low, high = nonzero.quantile(1 / 3), nonzero.quantile(2 / 3)
        combined.loc[ai[(ai > low) & (ai <= high)].index, tier_col] = "Moderate Access"
        combined.loc[ai[ai > high].index, tier_col] = "Served"

print("THREE-TIER VALUES:")
for tier_col in TIER_AI:
    print(combined[tier_col].value_counts().to_string())
    print()


In [ ]:
# SIX-CLASS TITLE-CASE MAP. The dictionary and the web app use title-case
# Access Typology labels; the table stores sentence case until this point.
TYPOLOGY_LABELS = {
    "Pharmacy desert": "Pharmacy Desert",
    "Connectivity gap": "Connectivity Gap",
    "Demand overcrowding": "Demand Overcrowding",
    "Access gap": "Access Gap",
    "Well-served": "Well-Served",
    "Artifact zone": "Artifact Zone",
}

for col in ("walk_typology", "drive_typology"):
    unknown = set(combined[col].dropna()) - set(TYPOLOGY_LABELS)
    if unknown:
        raise ValueError(f"Unexpected {col} values: {sorted(unknown)}")
    combined[col] = combined[col].map(TYPOLOGY_LABELS)

print("TYPOLOGY LABELS MAPPED TO TITLE CASE")


In [ ]:
# CODE COLUMNS: round then cast to nullable integers (Int64) so codes
# serialize without a decimal point while nulls survive.
CODE_COLUMNS = ["EA_CODE", "WardID", "SP_CODE", "MP_CODE", "MN_CODE", "DC_MN_C"]

for col in CODE_COLUMNS:
    combined[col] = combined[col].round().astype("Int64")

print("CODE COLUMNS CAST:", CODE_COLUMNS)


In [ ]:
# ORDER BY THE DICTIONARY'S ROW ORDER, THEN RENAME TO ITS FIELD NAMES.
keep_order = dictionary["full_name"].tolist()
extra = [c for c in combined.columns if c not in keep_order]  # none expected; geometry joins below

missing = sorted(set(keep_order) - set(combined.columns))
unexpected = sorted(set(combined.columns) - set(keep_order))
if missing or unexpected:
    raise ValueError(
        f"Column ordering check failed.\n"
        f"  missing: {missing}\n  not in dictionary: {unexpected}"
    )

rename = dictionary.set_index("full_name")["field"].to_dict()
final_df = combined[keep_order + extra].rename(columns = rename)

# JOIN GEOMETRY ONE-TO-ONE ON EA_CODE FROM THE DEDUPLICATED SAL LAYER.
if sal_geo["EA_CODE"].duplicated().any():
    raise ValueError(
        f"SAL layer has {sal_geo['EA_CODE'].duplicated().sum():,} duplicate EA_CODE values"
    )

final_gdf = gpd.GeoDataFrame(
    sal_geo[["EA_CODE", "geometry"]].merge(
        final_df, on = "EA_CODE", how = "inner", validate = "1:1"
    ),
    geometry = "geometry",
    crs = sal_geo.crs,
)

if len(final_gdf) != 38380:
    raise ValueError(f"Expected 38,380 rows after the geometry join, got {len(final_gdf):,}")

if final_gdf.crs is not None and final_gdf.crs.to_epsg() != 4326:
    final_gdf = final_gdf.to_crs(4326)

print(f"FINAL DATASET: {len(final_gdf):,} rows, {len(final_gdf.columns) - 1} fields")
print(f"CRS: {final_gdf.crs}")


## EXPORT FINAL DATASET

One code cell writes both outputs: the GeoJSON in EPSG:4326 with
6-decimal coordinate precision, and the CSV with the same attributes in
the same order, without geometry.


In [ ]:
# WRITE THE FINAL DATASET AS GEOJSON AND CSV.
OUT_SAL_COMBINED_GEOJSON.parent.mkdir(parents = True, exist_ok = True)

final_gdf.to_file(OUT_SAL_COMBINED_GEOJSON, driver = "GeoJSON", COORDINATE_PRECISION = 6)
final_gdf.drop(columns = "geometry").to_csv(OUT_SAL_COMBINED_CSV, index = False)

print(f"GEOJSON SAVED: {OUT_SAL_COMBINED_GEOJSON}")
print(f"CSV SAVED: {OUT_SAL_COMBINED_CSV}")
print(f"Rows: {len(final_gdf):,}, Fields: {len(final_gdf.columns) - 1}")


## PHARMACY POINTS

The master pharmacy list's points in EPSG:4326, filtered to those that
fall within a study-area SAL of the deduplicated layer (the same within
rule the 2SFCA notebook uses). The master list's original 22 columns keep
their order, with geometry appended.


In [ ]:
# BUILD PHARMACY POINTS: master list coordinates in EPSG:4326, filtered
# to points that fall within a study-area SAL of the deduplicated layer.
pharmacy_raw = pd.read_csv(IN_PHARMACIES)
pharmacy_points = gpd.GeoDataFrame(
    pharmacy_raw,
    geometry = gpd.points_from_xy(pharmacy_raw["LNG"], pharmacy_raw["LAT"]),
    crs = "EPSG:4326",
)

sal_4326 = sal_geo.to_crs("EPSG:4326")
pharmacy_points = gpd.sjoin(
    pharmacy_points, sal_4326[["EA_CODE", "geometry"]], predicate = "within"
)

# Keep the master list's original 22 columns in order, plus geometry.
pharmacy_points = pharmacy_points[list(pharmacy_raw.columns) + ["geometry"]]

if len(pharmacy_points) != 2152:
    raise ValueError(f"Expected 2,152 pharmacy points, got {len(pharmacy_points):,}")

OUT_PHARMACIES_GEOJSON.parent.mkdir(parents = True, exist_ok = True)
pharmacy_points.to_file(OUT_PHARMACIES_GEOJSON, driver = "GeoJSON")

print(f"PHARMACY POINTS SAVED: {OUT_PHARMACIES_GEOJSON}")
print(f"Points: {len(pharmacy_points):,}")


## KEY FINDINGS

### The equity gap in three numbers.

**7.8 million people in KZN (62.8% of the province) live in a walking
Pharmacy Desert** where zero pharmacies are reachable within 3 km on the
pedestrian network. In Gauteng, 2.4 million (15.8%) are in the same
condition. The 47-point gap between provinces is the spatial legacy of
apartheid-era infrastructure investment.

**Even by car, 4.4 million KZN residents (35.8%) remain in a Pharmacy
Desert** at the 10 km drive threshold. In Gauteng, this drops to
179,000 (1.2%). Driving does not solve KZN's access problem as it looks like it
merely shifts the threshold at which the problem becomes visible.

**The "Connectivity Gap" category is almost exclusively a Gauteng
phenomenon** (2,795 SALs walking, 1,275 driving). These are SALs where
a pharmacy exists nearby in Euclidean terms, but the walk or drive
network does not connect the centroid to it. This is a data quality
signal of the OSM coverage gap rather than a genuine access barrier, and it
affects 14.1% of Gauteng's population on the walk network.

### Snap distance validates the uncertainty pattern

**36.3% of "Pharmacy Desert" SALs on the drive network have a snap flag**
(centroid >500m from any drive-network node), compared to just 1.1% of
"Well-Served" SALs. It looks like the category that matters most for NHI
policy is also the category with the least reliable metrics. Pharmacy Desert
classifications for snap-flagged SALs should carry a data quality asterisk
in any policy communication.

### The redundancy deficit

The gap between k=1 and k=3 threshold exceedance rates measures how
many SALs have fragile, single-pharmacy access. In Gauteng, 16.5% of
SALs exceed 3 km walk at k=1, rising to 39.1% at k=3, which is a 22.6-point
redundancy deficit, meaning roughly one in four Gauteng SALs has one
pharmacy within reach but not three. In KZN, the baseline is already
so high (61.3% at k=1) that the deficit is smaller in relative terms
(43.8% at k=3), but the absolute numbers note that over 5.7
million people lack three walking-distance pharmacies.

### Artifact Zone interpretation

**17.3% of KZN SALs (1.83 million people) are classified as "Artifact Zone"
on the walk network** where nonzero $A_{i}$ scores appear despite being beyond the 3 km
walk threshold. These are mechanically driven by low local demand
(industrial, sparsely populated rural) producing inflated Ai scores.
The walk_typology column correctly flags these so they are not mistaken
for genuinely well-served areas.

## COLUMN DESCRIPTIONS

**93 columns. Organized by analytical priority.**

### Identifiers (cols 0–5)

| Column | Type | Description |
|---|---|---|
| `EA_CODE` | Int64 | Stats SA Enumeration Area code. Unique SAL identifier across all pipeline outputs. |
| `PR_NAME` | str | Province name: Gauteng or KwaZulu-Natal. |
| `WardID` | int64 | 2020 municipal ward code for the ward containing the largest share of this SAL's area. |
| `EA_GTYPE` | str | Settlement geography type: Urban, Traditional, or Farms. |
| `EA_TYPE` | str | Settlement classification: Suburb, Township, Traditional residential, Industrial, Vacant, Informal residential, Small holdings, Collective living quarters, Parks and recreation, Commercial. "Formal residential" renamed to "Suburb" for interpretability. |
| `econ_status` | str | Economic status from 2011 census land classification: Wealthy, Non_Wealthy, or Non_Residential. 1,289 nulls for unclassified SALs. |

### Geographic hierarchy (cols 6–16)

Sub-place through district council codes from the Stats SA SAL shapefile. Enables aggregation to any level of the municipal geography hierarchy without a separate join.

| Column | Type | Description |
|---|---|---|
| `SP_CODE` | str | Sub-place code. |
| `SP_NAME` | str | Sub-place name. |
| `MP_CODE` | str | Main place code. |
| `MP_NAME` | str | Main place name. |
| `MN_MDB_C` | str | Municipality MDB code. |
| `MN_CODE` | str | Municipality code. |
| `MN_NAME` | str | Municipality name. |
| `MN_TYPE` | str | Municipality type (e.g., Category A, B, or C). |
| `DC_MDB_C` | str | District council MDB code. |
| `DC_MN_C` | str | District council municipality code. |
| `DC_NAME` | str | District council name. |

### Population (cols 17–20)

| Column | Type | Description |
|---|---|---|
| `sal2011_pop` | float64 | 2011 census population for this SAL. Includes imputed values (houses × 3.3) for SALs with zero census population but nonzero dwelling count (1,190 SALs). |
| `sal2023_est` | float64 | Estimated 2023 population from the proportional step-down model: ward 2023 pop × (SAL 2011 share of ward 2011 pop). |
| `growth_rate` | float64 | Compound annual growth rate 2011–2023. NaN for 1,270 SALs with zero population in both years. |
| `area_km2` | float64 | SAL polygon area in square kilometers, computed from Shape_Area in EPSG:32735. |

### 2011 census demographics (cols 21–33)

2,084 SALs have null values across all demographic columns (non-residential or
unmatched SALs absent from the 2011 census release).

| Column | Type | Description |
|---|---|---|
| `Black_African` | float64 | 2011 census count of Black African population. Renamed from shapefile-truncated `Black_Afri`. |
| `White` | float64 | 2011 census count of White population. |
| `Coloured` | float64 | 2011 census count of Coloured population. |
| `Indian_Asian` | float64 | 2011 census count of Indian or Asian population. The census does not disaggregate these groups; this is a combined category. Renamed from shapefile-truncated `Indian_or`. |
| `Other` | float64 | 2011 census count of Other population group. |
| `pop_0_14` | float64 | 2011 census population aged 0–14. Pediatric, caregiver is the pharmacy user. Derived by summing F0_4 + F5_9 + F10_14. |
| `pop_15_34` | float64 | Population aged 15–34. Young adult baseline demand. Derived from F15_19 + F20_24 + F25_29 + F30_34. |
| `pop_35_49` | float64 | Population aged 35–49. Chronic disease onset, rising prescription dependency. |
| `pop_50_64` | float64 | Population aged 50–64. Highest working-age pharmacy dependency. |
| `pop_65plus` | float64 | Population aged 65+. Highest medication need and greatest travel difficulty. |
| `pop_total_2011` | float64 | Sum of all 17 original age cohorts. Cross-check against `sal2011_pop`; diverges for 1,190 imputed SALs where `sal2011_pop` includes the houses × 3.3 adjustment but age cohorts do not. |
| `num_houses` | float64 | 2011 census dwelling count for this SAL. Used in the zero-population imputation step. |
| `num_build` | float64 | Google Open Buildings v3 building footprint count. Foundation for the dasymetric weighting exploration (not yet integrated into the population model). |

### Primary outputs: typology, 2SFCA scores, and transforms (cols 34–41)

| Column | Type | Description |
|---|---|---|
| `walk_typology` | str | Six-category access classification combining Ai_walk with walk_dist_k1_km at the 3 km policy threshold. See typology definitions below. |
| `drive_typology` | str | Same classification for drive mode at the 10 km policy threshold. |
| `Ai_walk` | float64 | Two-Step Floating Catchment Area accessibility score on the pedestrian network. Higher = better access. Zero means no pharmacy reachable within the catchment radius. Uses negative exponential decay (β = 0.0003) and province-specific catchments (Gauteng: 2 km, KZN: 3 km). |
| `Ai_drive` | float64 | 2SFCA accessibility score on the drive network. Same decay function, with catchments of 5 km (Gauteng) and 10 km (KZN). |
| `walk_rank` | float64 | Province-specific percentile rank of Ai_walk (0–1). More interpretable than raw Ai for communication and choropleth mapping. |
| `drive_rank` | float64 | Province-specific percentile rank of Ai_drive (0–1). |
| `walk_log` | float64 | Natural log transform of Ai_walk. Compresses the right tail for visualization; zero-Ai SALs remain at zero. |
| `drive_log` | float64 | Natural log transform of Ai_drive. |

**Typology definitions:**

| Category | $A_i$ score | Distance | Interpretation |
|---|---|---|---|
| Well-served | Top tercile (nonzero) | < threshold | Pharmacy nearby with favorable supply-to-demand ratio. |
| Demand overcrowding | Bottom tercile (nonzero) | < threshold | Pharmacy nearby but serving too large a population. Candidate for NHI capacity investment at the existing site. |
| Connectivity gap | Zero | < threshold | Pharmacy exists nearby in Euclidean terms but is network-unreachable. Likely an OSM coverage gap or catchment boundary artifact. |
| Access gap | Bottom tercile (nonzero) | ≥ threshold | Far from a pharmacy, and that pharmacy is also under-supplied relative to demand. Secondary infrastructure target. |
| Pharmacy desert | Zero | ≥ threshold | No pharmacy reachable on the network and nearest pharmacy is also far in absolute terms. Primary NHI infrastructure target. |
| Artifact zone | Top tercile (nonzero) | ≥ threshold | High $A_i$ mechanically driven by low local demand (industrial, sparsely populated rural). Score is not policy-relevant; use distance columns as primary indicator. |

Terciles are computed province-specifically from nonzero $A_i$ values only, preventing
zero-inflation from collapsing the boundary toward zero. Walk and drive terciles are
computed independently.

### k=1 distances and circuity (cols 42–46)

k=1 answers the reachability question: can any pharmacy be reached at all?

| Column | Type | Description |
|---|---|---|
| `walk_dist_k1_km` | float64 | Shortest-path distance (km) on the OSM pedestrian network to the nearest pharmacy, via Dijkstra. NaN if no path found within the 50 km cutoff (4,931 SALs, 12.8%). |
| `drive_dist_k1_km` | float64 | Same on the OSM drive network. Graph converted to undirected for routing. NaN for 5,001 SALs (13.0%). |
| `euclidean_dist_k1_km` | float64 | Straight-line distance (km) to nearest pharmacy. Always non-null; used as a lower bound and baseline for circuity. |
| `walk_circuity_k1` | float64 | walk_dist_k1_km / euclidean_dist_k1_km. Values of 1.2–1.5 are typical urban grids; >3 suggests network gaps or major barriers. |
| `drive_circuity_k1` | float64 | Same ratio for the drive network. |

### k=2 distances (cols 47–49)

k=2 answers the redundancy question: is there a fallback pharmacy?

| Column | Type | Description |
|---|---|---|
| `walk_dist_k2_km` | float64 | Walk network distance (km) to the 2nd nearest pharmacy. NaN for 8,681 SALs (22.6%). |
| `drive_dist_k2_km` | float64 | Drive network distance (km) to the 2nd nearest pharmacy. NaN for 8,660 SALs (22.6%). |
| `euclidean_dist_k2_km` | float64 | Straight-line distance (km) to the 2nd nearest pharmacy. Always non-null. |

### k=3 distances (cols 50–52)

k=3 answers the choice question: do residents have meaningful pharmacy options?

| Column | Type | Description |
|---|---|---|
| `walk_dist_k3_km` | float64 | Walk network distance (km) to the 3rd nearest pharmacy. NaN for 9,689 SALs (25.2%). |
| `drive_dist_k3_km` | float64 | Drive network distance (km) to the 3rd nearest pharmacy. NaN for 9,679 SALs (25.2%). |
| `euclidean_dist_k3_km` | float64 | Straight-line distance (km) to the 3rd nearest pharmacy. Always non-null. |

### Snap distances and data quality flags (cols 53–56)

| Column | Type | Description |
|---|---|---|
| `walk_snap_dist_m` | float64 | Euclidean distance (meters) from the SAL centroid to the nearest pedestrian network node. Measures how far the Dijkstra origin was displaced from the true centroid. |
| `drive_snap_dist_m` | float64 | Same for the drive network. Values differ because walk and drive networks have different node sets. |
| `walk_snap_flag` | bool | True if walk_snap_dist_m > 500m. Scores for flagged SALs carry elevated measurement uncertainty. |
| `drive_snap_flag` | bool | True if drive_snap_dist_m > 500m. |

### Threshold exceedance flags w/ k=1 (cols 57–68)

| Column pattern | Type | Description |
|---|---|---|
| `exceeds_walk_k1_{1,2,3,5}km` | bool | True if nearest pharmacy exceeds the walk threshold. NaN walk distances treated as exceeding. |
| `exceeds_drive_k1_{5,10,15,20}km` | bool | True if nearest pharmacy exceeds the drive threshold. |
| `exceeds_euclidean_k1_{1,3,5,10}km` | bool | True if straight-line distance to nearest pharmacy exceeds the threshold. Always non-null. |

### Threshold exceedance flags w/ k=2 (cols 69–80)

Same structure as k=1, applied to the 2nd nearest pharmacy. The gap between
k=1 and k=2 exceedance rates at the same threshold is the **redundancy deficit**.

### Threshold exceedance flags w/ k=3 (cols 81–92)

Same structure, applied to the 3rd nearest pharmacy. k=3 exceedance measures
whether a SAL has meaningful pharmacy choice or depends on a single facility.

### Centroid coordinates (cols 93–94)

| Column | Type | Description |
|---|---|---|
| `centroid_lat` | float64 | Latitude of SAL geometric centroid (WGS84 / EPSG:4326). |
| `centroid_lng` | float64 | Longitude of SAL geometric centroid (WGS84 / EPSG:4326). |


## COLUMNS DROPPED AND EXCLUDED

### Dropped during reorganization (9 columns: 102 to 93)

All dropped columns are redundant unit conversions. Each has a kilometers
counterpart retained in the table.

| Dropped | Retained equivalent |
|---|---|
| `euclidean_dist_k{1,2,3}_m` | `euclidean_dist_k{1,2,3}_km` |
| `walk_dist_k{1,2,3}_m` | `walk_dist_k{1,2,3}_km` |
| `drive_dist_k{1,2,3}_m` | `drive_dist_k{1,2,3}_km` |

Snap distance columns (`walk_snap_dist_m`, `drive_snap_dist_m`) are retained
in meters because the 500m flag threshold requires sub-kilometer resolution.

### Excluded during initial column selection (never carried forward)

These columns existed in the `all_access_w_snap.csv` source
but were not included in `access_cols`. They are recoverable by joining on
`EA_CODE` to the indicated source file.

| Column(s) | Category | Why excluded |
|---|---|---|
| `PR_MDB_C`, `PR_CODE`, `MD_CODE`, `MD_NAME` | Geographic hierarchy (remaining) | Province MDB code, province code, and metro district codes. Province name captured in `PR_NAME`. Recoverable from SAL shapefile. |
| `SAL_CODE` | Alternate ID | Redundant with `EA_CODE`. |
| `OBJECTID_1`, `OBJECTID_2`, `OBJECTID_3` | Join artifacts | ArcGIS auto-generated row IDs. |
| `EA_CODE_1`, `EA_CODE_12`, `EA_CODE_13` | Join artifacts | Duplicate EA_CODE columns from successive spatial joins. |
| `census_war` | Join artifact | Shapefile-truncated ward code; clean version is `WardID`. |
| `AREA`, `PERCENTAGE`, `FREQUENCY`, `MAX_AREA` | Join artifacts | Intermediate spatial join metrics. Ward assignment captured in `WardID`. |
| `sal2011_po`, `sal_pop_de` | Truncated duplicates | `sal2011_po` is the DBF-truncated `sal2011_pop`. `sal_pop_de` is raw density, re-derivable as `sal2011_pop / area_km2`. |
| `Shape_Leng`, `Shape_Le_1`, `Shape_Area` | Raw geometry fields | `area_km2` is the clean derived equivalent. |
| `ALBERS_ARE`, `EA_area_km` | Redundant area | Alternative area calculations superseded by `area_km2`. |
| `F4_class`, `F4_class_2` | Renamed source | Original column names for `econ_status`. |
| `old_EA_TYP` | Deprecated label | Legacy EA type classification superseded by `EA_TYPE`. |
| `url` | Reference link | Census 2011 Adrian Frith lookup URL. Not analytical. |
| `smallplace` | Place label | Descriptive small-place name. Recoverable from SAL shapefile. |
| `population` | Alternate pop count | 2011 total from original census shapefile (pre-imputation). Cross-checkable via `pop_total_2011`. |
| `ward2023_pop` | Model intermediate | 2023 ward total (step-down numerator). Source: `pop_pred_final.csv`. |
| `ward2011_sum` | Model intermediate | Sum of 2011 SAL pops within ward (step-down denominator). Source: `pop_pred_final.csv`. |
| `share2011`, `dasym_weight` | Model intermediate | Identical columns, SAL share of ward 2011 pop. Source: `pop_pred_final.csv`. |
| `sal_dense`, `log_density` | EDA variables | Population density and log transform from the step-down notebook. |
| `node` | Routing debug | OSM network node ID for centroid snap. Useful for reproducing Dijkstra runs only. |
| `F0_4` through `F85_` (17 cols) | Collapsed into age bands | Replaced by `pop_0_14`, `pop_15_34`, `pop_35_49`, `pop_50_64`, `pop_65plus`. |
